<a href="https://colab.research.google.com/github/Noors-lab/Model-s_summaries/blob/main/training_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.7 MB/s eta 0:00:00


# version 1

In [8]:
import json
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# ---------- 1. Load + standardize ----------
DATA_DIR = '/content/drive/MyDrive/VigIQ/final_dataset'

def load_json(name):
    with open(f'{DATA_DIR}/{name}') as f:
        return json.load(f)

train_data = load_json('train.json')
val_data   = load_json('val.json')
test_data  = load_json('test.json')

X_mean = np.load(f'{DATA_DIR}/X_mean_normal_v1.npy')
X_std  = np.load(f'{DATA_DIR}/X_std_normal_v1.npy')

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

class SeqDataset(Dataset):
    def __init__(self, data, mean, std):
        self.data, self.mean, self.std = data, mean, std
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        kp = np.array(item['keypoints'], dtype=np.float32)
        kp = (kp - self.mean) / self.std
        return torch.tensor(kp, dtype=torch.float32), float(item['label'])

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([s.shape[0] for s in seqs])
    padded = pad_sequence(seqs, batch_first=True)
    return padded, lengths, torch.tensor(labels, dtype=torch.float32)

train_loader = DataLoader(SeqDataset(train_data, X_mean, X_std), batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(SeqDataset(val_data, X_mean, X_std), batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(SeqDataset(test_data, X_mean, X_std), batch_size=64, shuffle=False, collate_fn=collate_fn)

# ---------- 2. Model ----------
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)  # last layer, fwd+bwd
        return self.classifier(h_final).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ShopliftingLSTM().to(device)

n_normal = sum(1 for d in train_data if d['label'] == 0)
n_not_normal = sum(1 for d in train_data if d['label'] == 1)
pos_weight = torch.tensor([n_normal / n_not_normal]).to(device)
print(f"pos_weight = {pos_weight.item():.2f}  (train: {n_normal} normal / {n_not_normal} not-normal)")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# ---------- 3. Train ----------
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, lengths, labels in loader:
            x, labels = x.to(device), labels.to(device)
            logits = model(x, lengths)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            all_preds.append(torch.sigmoid(logits).detach().cpu())
            all_labels.append(labels.cpu())
    return total_loss / len(loader.dataset), torch.cat(all_preds), torch.cat(all_labels)

best_val_loss, best_state, patience, patience_ctr = float('inf'), None, 8, 0
for epoch in range(50):
    train_loss, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_preds, val_labels = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss, best_state, patience_ctr = val_loss, copy.deepcopy(model.state_dict()), 0
    else:
        patience_ctr += 1
    print(f"Epoch {epoch+1:2d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | best {best_val_loss:.4f}")
    if patience_ctr >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state)
print(f"\nLoaded best model (val_loss={best_val_loss:.4f})")

# ---------- 4. Threshold sweep (val set only) ----------
_, val_probs, val_labels = run_epoch(val_loader, train=False)
val_probs, val_labels = val_probs.numpy(), val_labels.numpy()

print(f"\n{'Threshold':>10} | {'Normal-Acc':>10} | {'NotNormal-Recall':>17} | {'Flagged (n)':>11}")
print("-" * 56)
sweep_results = []
for t in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
    preds = (val_probs >= t).astype(int)
    normal_mask, not_normal_mask = val_labels == 0, val_labels == 1
    normal_acc = (preds[normal_mask] == 0).mean()
    not_normal_recall = (preds[not_normal_mask] == 1).mean()
    n_flagged = int(preds.sum())
    sweep_results.append((t, normal_acc, not_normal_recall, n_flagged))
    print(f"{t:>10.2f} | {normal_acc*100:>9.2f}% | {not_normal_recall*100:>16.2f}% | {n_flagged:>11d}")

Train: 13043 | Val: 2794 | Test: 2796
pos_weight = 22.42  (train: 12486 normal / 557 not-normal)
Epoch  1 | train_loss 0.3954 | val_loss 0.2504 | best 0.2504
Epoch  2 | train_loss 0.2749 | val_loss 0.2850 | best 0.2504
Epoch  3 | train_loss 0.3850 | val_loss 0.3894 | best 0.2504
Epoch  4 | train_loss 0.2870 | val_loss 0.2437 | best 0.2437
Epoch  5 | train_loss 0.2707 | val_loss 0.2320 | best 0.2320
Epoch  6 | train_loss 0.2337 | val_loss 0.2161 | best 0.2161
Epoch  7 | train_loss 0.2419 | val_loss 0.2117 | best 0.2117
Epoch  8 | train_loss 0.2344 | val_loss 0.2556 | best 0.2117
Epoch  9 | train_loss 0.2219 | val_loss 0.2293 | best 0.2117
Epoch 10 | train_loss 0.2217 | val_loss 0.2205 | best 0.2117
Epoch 11 | train_loss 0.2222 | val_loss 0.2225 | best 0.2117
Epoch 12 | train_loss 0.2463 | val_loss 0.2112 | best 0.2112
Epoch 13 | train_loss 0.2472 | val_loss 0.2329 | best 0.2112
Epoch 14 | train_loss 0.2182 | val_loss 0.2359 | best 0.2112
Epoch 15 | train_loss 0.2328 | val_loss 0.2180 | 

# version 2

In [23]:
import json
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

# ---------- 1. Load balanced dataset + split ----------
BALANCED_PATH = '/content/drive/MyDrive/VigIQ/merged_dataset/balanced_dataset.json'

with open(BALANCED_PATH) as f:
    all_data = json.load(f)

labels = [d['label'] for d in all_data]
train_data, temp_data = train_test_split(all_data, test_size=0.30, stratify=labels, random_state=42)
temp_labels = [d['label'] for d in temp_data]
val_data, test_data = train_test_split(temp_data, test_size=0.50, stratify=temp_labels, random_state=42)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print(f"Train NOT-NORMAL: {sum(d['label']==1 for d in train_data)}")
print(f"Val NOT-NORMAL:   {sum(d['label']==1 for d in val_data)}")
print(f"Test NOT-NORMAL:  {sum(d['label']==1 for d in test_data)}")

# ---------- 2. Compute standardization stats from TRAIN only ----------
all_frames = np.concatenate([np.array(d['keypoints'], dtype=np.float32) for d in train_data], axis=0)
X_mean = all_frames.mean(axis=0)
X_std = all_frames.std(axis=0)
X_std[X_std < 1e-6] = 1.0  # avoid div-by-zero on constant dims

SAVE_DIR = '/content/drive/MyDrive/VigIQ/normal_vs_notnormal_v2'
import os
os.makedirs(SAVE_DIR, exist_ok=True)
np.save(f'{SAVE_DIR}/X_mean_v2.npy', X_mean)
np.save(f'{SAVE_DIR}/X_std_v2.npy', X_std)
print(f"Saved new mean/std to {SAVE_DIR}")

# ---------- 3. Dataset / DataLoader ----------
class SeqDataset(Dataset):
    def __init__(self, data, mean, std):
        self.data, self.mean, self.std = data, mean, std
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        kp = np.array(item['keypoints'], dtype=np.float32)
        kp = (kp - self.mean) / self.std
        return torch.tensor(kp, dtype=torch.float32), float(item['label'])

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([s.shape[0] for s in seqs])
    padded = pad_sequence(seqs, batch_first=True)
    return padded, lengths, torch.tensor(labels, dtype=torch.float32)

train_loader = DataLoader(SeqDataset(train_data, X_mean, X_std), batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(SeqDataset(val_data, X_mean, X_std), batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(SeqDataset(test_data, X_mean, X_std), batch_size=64, shuffle=False, collate_fn=collate_fn)

# ---------- 4. Model ----------
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.classifier(h_final).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ShopliftingLSTM().to(device)

n_normal = sum(1 for d in train_data if d['label'] == 0)
n_not_normal = sum(1 for d in train_data if d['label'] == 1)
pos_weight = torch.tensor([n_normal / n_not_normal]).to(device)
print(f"pos_weight = {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# ---------- 5. Train ----------
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, lengths, labels_b in loader:
            x, labels_b = x.to(device), labels_b.to(device)
            logits = model(x, lengths)
            loss = criterion(logits, labels_b)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            all_preds.append(torch.sigmoid(logits).detach().cpu())
            all_labels.append(labels_b.cpu())
    return total_loss / len(loader.dataset), torch.cat(all_preds), torch.cat(all_labels)

best_val_loss, best_state, patience, patience_ctr = float('inf'), None, 8, 0
for epoch in range(50):
    train_loss, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_preds, val_labels_t = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss, best_state, patience_ctr = val_loss, copy.deepcopy(model.state_dict()), 0
    else:
        patience_ctr += 1
    print(f"Epoch {epoch+1:2d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | best {best_val_loss:.4f}")
    if patience_ctr >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state)
torch.save(model.state_dict(), f'{SAVE_DIR}/model_v2.pth')
print(f"\nBest model saved (val_loss={best_val_loss:.4f})")

# ---------- 6. Threshold sweep ----------
_, val_probs, val_labels_np = run_epoch(val_loader, train=False)
val_probs, val_labels_np = val_probs.numpy(), val_labels_np.numpy()

print(f"\n{'Threshold':>10} | {'Normal-Acc':>10} | {'NotNormal-Recall':>17} | {'Flagged (n)':>11}")
print("-" * 56)
for t in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
    preds = (val_probs >= t).astype(int)
    normal_mask, not_normal_mask = val_labels_np == 0, val_labels_np == 1
    normal_acc = (preds[normal_mask] == 0).mean()
    not_normal_recall = (preds[not_normal_mask] == 1).mean()
    print(f"{t:>10.2f} | {normal_acc*100:>9.2f}% | {not_normal_recall*100:>16.2f}% | {int(preds.sum()):>11d}")

# ---------- 7. THE CRITICAL CHECK: source-shortcut diagnostic ----------
val_sources = [d['source'] for d in val_data]
val_source_prefix = np.array(['retails' if s.startswith('retails') else 'dataset1' for s in val_sources])

print(f"\n--- Source-shortcut check (this is what actually tells us if it's fixed) ---")
for prefix in ['retails', 'dataset1']:
    mask = val_source_prefix == prefix
    for true_label, name in [(0, 'NORMAL'), (1, 'NOT-NORMAL')]:
        sub_mask = mask & (val_labels_np == true_label)
        n = sub_mask.sum()
        if n > 0:
            print(f"{prefix:<10} | true={name:<10} | n={n:>4} | mean_prob={val_probs[sub_mask].mean():.4f}")

Train: 1818 | Val: 390 | Test: 390
Train NOT-NORMAL: 606
Val NOT-NORMAL:   130
Test NOT-NORMAL:  130
Saved new mean/std to /content/drive/MyDrive/VigIQ/normal_vs_notnormal_v2
pos_weight = 2.00
Epoch  1 | train_loss 0.8046 | val_loss 0.8021 | best 0.8021
Epoch  2 | train_loss 0.7767 | val_loss 0.7929 | best 0.7929
Epoch  3 | train_loss 0.7596 | val_loss 0.8014 | best 0.7929
Epoch  4 | train_loss 0.7893 | val_loss 0.7988 | best 0.7929
Epoch  5 | train_loss 0.7587 | val_loss 0.8159 | best 0.7929
Epoch  6 | train_loss 0.7773 | val_loss 0.7893 | best 0.7893
Epoch  7 | train_loss 0.7439 | val_loss 0.8250 | best 0.7893
Epoch  8 | train_loss 0.7533 | val_loss 0.8042 | best 0.7893
Epoch  9 | train_loss 0.7383 | val_loss 0.8377 | best 0.7893
Epoch 10 | train_loss 0.7439 | val_loss 0.8030 | best 0.7893
Epoch 11 | train_loss 0.7490 | val_loss 0.7972 | best 0.7893
Epoch 12 | train_loss 0.7331 | val_loss 0.7904 | best 0.7893
Epoch 13 | train_loss 0.7153 | val_loss 0.8008 | best 0.7893
Epoch 14 | tra

# version 3


In [24]:
import json
import copy
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

# ---------- 1. Load balanced_v2 dataset + split ----------
BALANCED_PATH = '/content/drive/MyDrive/VigIQ/merged_dataset/balanced_dataset_v2.json'

with open(BALANCED_PATH) as f:
    all_data = json.load(f)

labels = [d['label'] for d in all_data]
train_data, temp_data = train_test_split(all_data, test_size=0.30, stratify=labels, random_state=42)
temp_labels = [d['label'] for d in temp_data]
val_data, test_data = train_test_split(temp_data, test_size=0.50, stratify=temp_labels, random_state=42)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print(f"Train NOT-NORMAL: {sum(d['label']==1 for d in train_data)} | Val NOT-NORMAL: {sum(d['label']==1 for d in val_data)} | Test NOT-NORMAL: {sum(d['label']==1 for d in test_data)}")

# ---------- 2. Standardization stats from TRAIN only ----------
all_frames = np.concatenate([np.array(d['keypoints'], dtype=np.float32) for d in train_data], axis=0)
X_mean = all_frames.mean(axis=0)
X_std = all_frames.std(axis=0)
X_std[X_std < 1e-6] = 1.0

SAVE_DIR = '/content/drive/MyDrive/VigIQ/normal_vs_notnormal_v3'
os.makedirs(SAVE_DIR, exist_ok=True)
np.save(f'{SAVE_DIR}/X_mean_v3.npy', X_mean)
np.save(f'{SAVE_DIR}/X_std_v3.npy', X_std)

# ---------- 3. Dataset / DataLoader ----------
class SeqDataset(Dataset):
    def __init__(self, data, mean, std):
        self.data, self.mean, self.std = data, mean, std
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        kp = np.array(item['keypoints'], dtype=np.float32)
        kp = (kp - self.mean) / self.std
        return torch.tensor(kp, dtype=torch.float32), float(item['label'])

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([s.shape[0] for s in seqs])
    padded = pad_sequence(seqs, batch_first=True)
    return padded, lengths, torch.tensor(labels, dtype=torch.float32)

train_loader = DataLoader(SeqDataset(train_data, X_mean, X_std), batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(SeqDataset(val_data, X_mean, X_std), batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(SeqDataset(test_data, X_mean, X_std), batch_size=64, shuffle=False, collate_fn=collate_fn)

# ---------- 4. Model ----------
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.classifier(h_final).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ShopliftingLSTM().to(device)

n_normal = sum(1 for d in train_data if d['label'] == 0)
n_not_normal = sum(1 for d in train_data if d['label'] == 1)
pos_weight = torch.tensor([n_normal / n_not_normal]).to(device)
print(f"pos_weight = {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# ---------- 5. Train ----------
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, lengths, labels_b in loader:
            x, labels_b = x.to(device), labels_b.to(device)
            logits = model(x, lengths)
            loss = criterion(logits, labels_b)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            all_preds.append(torch.sigmoid(logits).detach().cpu())
            all_labels.append(labels_b.cpu())
    return total_loss / len(loader.dataset), torch.cat(all_preds), torch.cat(all_labels)

best_val_loss, best_state, patience, patience_ctr = float('inf'), None, 8, 0
for epoch in range(50):
    train_loss, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_preds, val_labels_t = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss, best_state, patience_ctr = val_loss, copy.deepcopy(model.state_dict()), 0
    else:
        patience_ctr += 1
    print(f"Epoch {epoch+1:2d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | best {best_val_loss:.4f}")
    if patience_ctr >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state)
torch.save(model.state_dict(), f'{SAVE_DIR}/model_v3.pth')
print(f"\nBest model saved (val_loss={best_val_loss:.4f})")

# ---------- 6. Threshold sweep ----------
_, val_probs, val_labels_np = run_epoch(val_loader, train=False)
val_probs, val_labels_np = val_probs.numpy(), val_labels_np.numpy()

print(f"\n{'Threshold':>10} | {'Normal-Acc':>10} | {'NotNormal-Recall':>17} | {'Flagged (n)':>11}")
print("-" * 56)
for t in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
    preds = (val_probs >= t).astype(int)
    normal_mask, not_normal_mask = val_labels_np == 0, val_labels_np == 1
    normal_acc = (preds[normal_mask] == 0).mean()
    not_normal_recall = (preds[not_normal_mask] == 1).mean()
    print(f"{t:>10.2f} | {normal_acc*100:>9.2f}% | {not_normal_recall*100:>16.2f}% | {int(preds.sum()):>11d}")

# ---------- 7. Source-shortcut diagnostic (the real check) ----------
val_sources = [d['source'] for d in val_data]
val_source_prefix = np.array(['retails' if s.startswith('retails') else 'dataset1' for s in val_sources])

print(f"\n--- Source-shortcut check ---")
for prefix in ['retails', 'dataset1']:
    mask = val_source_prefix == prefix
    for true_label, name in [(0, 'NORMAL'), (1, 'NOT-NORMAL')]:
        sub_mask = mask & (val_labels_np == true_label)
        n = sub_mask.sum()
        if n > 0:
            print(f"{prefix:<10} | true={name:<10} | n={n:>4} | mean_prob={val_probs[sub_mask].mean():.4f}")

Train: 1330 | Val: 285 | Test: 286
Train NOT-NORMAL: 606 | Val NOT-NORMAL: 130 | Test NOT-NORMAL: 130
pos_weight = 1.19
Epoch  1 | train_loss 0.7525 | val_loss 0.7483 | best 0.7483
Epoch  2 | train_loss 0.7395 | val_loss 0.7247 | best 0.7247
Epoch  3 | train_loss 0.7337 | val_loss 0.7348 | best 0.7247
Epoch  4 | train_loss 0.7322 | val_loss 0.7346 | best 0.7247
Epoch  5 | train_loss 0.7307 | val_loss 0.7376 | best 0.7247
Epoch  6 | train_loss 0.7301 | val_loss 0.7459 | best 0.7247
Epoch  7 | train_loss 0.7165 | val_loss 0.7327 | best 0.7247
Epoch  8 | train_loss 0.7203 | val_loss 0.7372 | best 0.7247
Epoch  9 | train_loss 0.7174 | val_loss 0.7278 | best 0.7247
Epoch 10 | train_loss 0.7246 | val_loss 0.7244 | best 0.7244
Epoch 11 | train_loss 0.7210 | val_loss 0.7315 | best 0.7244
Epoch 12 | train_loss 0.7106 | val_loss 0.7258 | best 0.7244
Epoch 13 | train_loss 0.7097 | val_loss 0.7156 | best 0.7156
Epoch 14 | train_loss 0.7022 | val_loss 0.7461 | best 0.7156
Epoch 15 | train_loss 0.67

In [26]:
import json

SAVE_DIR = '/content/drive/MyDrive/VigIQ/v3_best'

# Save the exact train/val/test split used for this model
with open(f'{SAVE_DIR}/train_data.json', 'w') as f:
    json.dump(train_data, f)
with open(f'{SAVE_DIR}/val_data.json', 'w') as f:
    json.dump(val_data, f)
with open(f'{SAVE_DIR}/test_data.json', 'w') as f:
    json.dump(test_data, f)

print(f"Saved exact train/val/test splits to {SAVE_DIR}")
print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print(f"Train NOT-NORMAL: {sum(d['label']==1 for d in train_data)}")

Saved exact train/val/test splits to /content/drive/MyDrive/VigIQ/v3_best
Train: 1330 | Val: 285 | Test: 286
Train NOT-NORMAL: 606
